# Review 2: controlled replay-label poisoning and label-consistency filtering

Import into Kaggle, enable **Internet** and **GPU (T4)**, attach the private `cic-collection.parquet`, and run top to bottom. This is a six-condition, single-seed balanced-subset experiment. After clean Task 1 training, only bounded fractions of **old-class replay labels** are altered before Task 2. Task 2 data, trusted validation, replay features, and starting weights stay clean. The filter is a simple baseline, not a backdoor or deployment defense. The combined flow collection has unverified per-row source provenance. This notebook has no saved execution results.


## 1. Pin the published source

Use the unified branch, not `main`; detach at the full source commit containing task construction, replay, label flips, teacher gate, and evaluation.


In [ ]:
from pathlib import Path
import subprocess, sys
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = 'c636331fc9cdf2361357fd901d9c93411e3f59ba'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-replay-poison')
if REPO_DIR.exists():
    raise RuntimeError('Repository folder already exists; use a fresh session or inspect before rerunning.')
subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
print('Verified source SHA:', checked_out_sha)


## 2. Check the live Kaggle runtime and attached input

Keep Kaggle's CUDA-enabled PyTorch; do not blindly install local pins. The preceding clean continual run used Python 3.13.15, PyTorch 2.11.0+cu128, NumPy 2.1.3, and PyArrow 23.0.1—different from earlier runs. Record the actual versions again. Set `PARQUET_PATH` if the attachment is ambiguous. The checksum must match the reviewed combined collection.


In [ ]:
from importlib.metadata import version
from dataclasses import asdict
from datetime import datetime, timezone
from math import ceil
from time import perf_counter
import hashlib, json, platform
import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib.pyplot as plt
import torch
from src.data import BROAD_CLASS_ORDER, sample_class_balanced_indices, stratified_split_indices, materialize_selected_rows
from src.continual_learning import TASK1_CLASS_IDS, TASK2_CLASS_IDS, prepare_two_task_dataset, select_balanced_replay
from src.models import ModelConfig, TabularTransformerClassifier
from src.poisoning import apply_label_flip
from src.mitigation import calibrate_label_consistency, apply_label_consistency_gate, label_inconsistency_scores
from src.evaluation import classification_metrics, forgetting_metrics, replay_gate_metrics
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required; enable Kaggle GPU and restart.')
assert pa.array([1, 2]).to_pylist() == [1, 2]
assert pd.Series([1, 2]).sum() == 3
assert torch.tensor([1.0], device='cuda').is_cuda
environment = {'python': platform.python_version(), 'packages': {name: version(name) for name in ('numpy', 'pandas', 'pyarrow', 'torch', 'matplotlib')}, 'torch_cuda_build': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0)}
print('Environment:', environment)
PARQUET_PATH = None  # e.g. '/kaggle/input/private-collection/cic-collection.parquet'
EXPECTED_INPUT_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
candidates = sorted(Path('/kaggle/input').rglob('cic-collection.parquet'))
print('Candidates:', [str(path) for path in candidates])
if PARQUET_PATH is None:
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one attached Parquet file, found {len(candidates)}; set PARQUET_PATH.')
    parquet_path = candidates[0]
else:
    parquet_path = Path(PARQUET_PATH)
    if not parquet_path.is_file() or not parquet_path.is_relative_to(Path('/kaggle/input')):
        raise ValueError('PARQUET_PATH must name an attached file under /kaggle/input')
start = perf_counter(); digest = hashlib.sha256()
with parquet_path.open('rb') as source_file:
    for chunk in iter(lambda: source_file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
input_sha256 = digest.hexdigest(); checksum_seconds = perf_counter() - start
print('Input bytes:', parquet_path.stat().st_size, 'SHA-256:', input_sha256)
if input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError('Attached file differs from the reviewed combined collection.')


## 3. Recreate the same clean task stream and replay buffer

The previous attachment contains a 500-row replay buffer but **not** a reusable Task 1 checkpoint. Train a new Task 1 model in this runtime; do not pretend its weights or metrics are the earlier run's. Sample 2,000 per broad class with seed 42 (the bounded sampler still scans labels), split 70/15/15, materialize selected rows, and fit preprocessing on Task 1 training rows only. Keep global IDs 0–7 and original row IDs. Replay selects 100 unique Task 1 training rows per old class, excluding validation/test.


In [ ]:
start = perf_counter(); sample = sample_class_balanced_indices(parquet_path, per_class_limit=2000, seed=42); sampling_seconds = perf_counter() - start
assert sample.class_order == BROAD_CLASS_ORDER
split = stratified_split_indices(sample.labels, seed=42)
start = perf_counter(); selected = materialize_selected_rows(parquet_path, sample.row_indices)
assert np.array_equal(selected.labels, sample.labels) and np.array_equal(selected.row_indices, sample.row_indices)
tasks = prepare_two_task_dataset(selected, split); preparation_seconds = perf_counter() - start
assert tasks.task1.class_ids == TASK1_CLASS_IDS and tasks.task2.class_ids == TASK2_CLASS_IDS
assert tasks.class_names == dict(enumerate(BROAD_CLASS_ORDER))
validation_rows = tasks.seen_validation.row_indices; test_rows = tasks.combined_test.row_indices
replay = select_balanced_replay(tasks.task1.train, class_ids=TASK1_CLASS_IDS, per_class=100, seed=42, forbidden_row_indices=np.concatenate((validation_rows, test_rows)))
assert replay.features.shape == (500, 54) and len(np.unique(replay.row_indices)) == 500
assert not np.intersect1d(replay.row_indices, validation_rows).size and not np.intersect1d(replay.row_indices, test_rows).size
for task_name in ('task1', 'task2'):
    task = getattr(tasks, task_name)
    for partition_name in ('train', 'validation', 'test'):
        part = getattr(task, partition_name)
        assert part.features.dtype == np.float32 and np.isfinite(part.features).all()
        assert set(np.unique(part.labels)) == set(task.class_ids)
        print(task_name, partition_name, part.features.shape, 'global IDs:', np.unique(part.labels).tolist())
print('Class mapping:', tasks.class_names, 'replay counts:', replay.class_counts)


## 4. Start an auditable output folder

Save the Git SHA, actual environment/input, sampled/split raw row IDs, frozen Task-1-only preprocessing state, and clean replay buffer. All generated files stay in `/kaggle/working`, not Git.


In [ ]:
OUTPUT_DIR = Path('/kaggle/working') / ('review2_replay_poison_mitigation_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
np.savez_compressed(OUTPUT_DIR / 'row_indices.npz', selected_original=sample.row_indices, train_positions=split.train, validation_positions=split.validation, test_positions=split.test, task1_train_original=tasks.task1.train.row_indices, task2_train_original=tasks.task2.train.row_indices, validation_original=validation_rows, test_original=test_rows)
np.savez_compressed(OUTPUT_DIR / 'clean_replay_buffer.npz', features=replay.features, labels=replay.labels, original_rows=replay.row_indices, task1_train_positions=replay.source_train_positions)
preprocessing = {'fit_scope': 'Task 1 training rows only', 'feature_names': list(tasks.preprocessor.feature_names), 'imputation_values': tasks.preprocessor.imputation_values.tolist(), 'means': tasks.preprocessor.means.tolist(), 'scales': tasks.preprocessor.scales.tolist()}
(OUTPUT_DIR / 'preprocessing.json').write_text(json.dumps(preprocessing, indent=2), encoding='utf-8')
manifest = {'purpose': 'controlled old-class replay-label corruption and frozen-teacher label-consistency gate; not Task 2 poisoning', 'source_sha': checked_out_sha, 'environment': environment, 'input': {'path': str(parquet_path), 'bytes': parquet_path.stat().st_size, 'sha256': input_sha256, 'provenance': 'combined flow collection; per-row source provenance unverified'}, 'sampling': {'per_class_limit': 2000, 'seed': 42, 'available_counts': sample.available_counts}, 'split': {'seed': 42, 'fractions': [0.70, 0.15, 0.15]}, 'class_names': {str(key): name for key, name in tasks.class_names.items()}, 'task1_class_ids': list(TASK1_CLASS_IDS), 'task2_class_ids': list(TASK2_CLASS_IDS), 'preprocessing_fit_scope': 'Task 1 training only', 'replay': {'seed': 42, 'per_old_class': 100, 'counts': replay.class_counts, 'total_rows': len(replay.labels)}, 'timings_seconds': {'checksum': checksum_seconds, 'sampling': sampling_seconds, 'materialization_and_preparation': preparation_seconds}, 'model_selection': 'Task 1: clean Task 1 validation; Task 2: same clean seen-class validation for all six arms; test excluded', 'conditions': {}}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Output folder:', OUTPUT_DIR)


## 5. Train a fresh clean Task 1 teacher and save its checkpoint

The attached run's checkpoint is unavailable. This fresh five-output model sees only Task 1 train/validation rows; IDs must remain 0–4. Save its best clean-validation-loss state, then reload independently as a frozen teacher. We will compare all six Task 2 conditions against this run's own Task 1 baseline, not the older observed numbers.


In [ ]:
model_config = ModelConfig(num_features=len(tasks.feature_names), num_classes=5, hidden_dim=64, num_heads=4, num_layers=2, mlp_dim=128, batch_size=128, learning_rate=0.001, epochs=20, early_stopping_patience=5, device='cuda', mixed_precision=False, seed=42)
manifest['task1_model_config'] = asdict(model_config)
task1_model = TabularTransformerClassifier(model_config, class_ids=TASK1_CLASS_IDS)
assert task1_model.class_ids == TASK1_CLASS_IDS
torch.cuda.synchronize(); start = perf_counter()
task1_history = task1_model.fit(tasks.task1.train.features, tasks.task1.train.labels, tasks.task1.validation.features, tasks.task1.validation.labels, verbose=True)
torch.cuda.synchronize(); task1_train_seconds = perf_counter() - start
assert task1_model.class_ids == TASK1_CLASS_IDS
assert task1_history.best_epoch == int(np.argmin(task1_history.validation_loss)) + 1
assert np.isfinite(task1_history.train_loss + task1_history.validation_loss + task1_history.validation_accuracy).all()
task1_checkpoint = OUTPUT_DIR / 'task1_best_validation.pt'; task1_model.save(task1_checkpoint)
task1_predictions = task1_model.predict(tasks.task1.test.features)
task1_before = classification_metrics(tasks.task1.test.labels, task1_predictions, class_ids=TASK1_CLASS_IDS, class_names=tasks.class_names, benign_class_id=0, source_class_ids=[1, 2, 3, 4], target_class_id=0)
teacher = TabularTransformerClassifier.load(task1_checkpoint, device='cuda')
teacher.network.eval()
assert teacher.class_ids == TASK1_CLASS_IDS and np.array_equal(teacher.predict(tasks.task1.test.features), task1_predictions)
task1_steps = ceil(len(tasks.task1.train.labels) / model_config.batch_size) * task1_history.epochs_completed
(OUTPUT_DIR / 'task1_history.json').write_text(json.dumps(task1_history.to_dict(), indent=2), encoding='utf-8')
(OUTPUT_DIR / 'task1_metrics.json').write_text(json.dumps(task1_before, indent=2), encoding='utf-8')
manifest['task1'] = {'train_rows': len(tasks.task1.train.labels), 'validation_rows': len(tasks.task1.validation.labels), 'test_rows': len(tasks.task1.test.labels), 'history': task1_history.to_dict(), 'train_seconds': task1_train_seconds, 'optimizer_steps': task1_steps, 'checkpoint': task1_checkpoint.name, 'reload_identical': True, 'test_metrics_before_task2': task1_before}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('New Task 1 best epoch:', task1_history.best_epoch, 'old-test accuracy:', task1_before['accuracy'])


## 6. Freeze the 95th-percentile clean validation threshold

For each supplied old-class label, score `1 - P_teacher(supplied label | flow)` using the teacher's explicit `class_ids` column mapping. Calibrate the **predeclared** 95th percentile on clean Task 1 validation scores, once, before creating attacked conditions. Scores **above** the threshold are quarantined; equality is retained. The teacher previously saw clean Task 1 distribution and is frozen. This threshold is not tuned against attack/test metrics. Do not use the old teacher on genuinely new Task 2 rows.


In [ ]:
validation_teacher_probs = teacher.predict_proba(tasks.task1.validation.features)
calibration = calibrate_label_consistency(validation_teacher_probs, tasks.task1.validation.labels, teacher.class_ids, quantile=0.95)
replay_teacher_probs = teacher.predict_proba(replay.features)
assert replay_teacher_probs.shape == (500, len(TASK1_CLASS_IDS))
assert np.isfinite(replay_teacher_probs).all() and np.allclose(replay_teacher_probs.sum(axis=1), 1.0, atol=1e-5)
calibration_record = asdict(calibration) | {'source': 'clean Task 1 validation only', 'teacher_class_ids': list(teacher.class_ids)}
(OUTPUT_DIR / 'calibration.json').write_text(json.dumps(calibration_record, indent=2), encoding='utf-8')
manifest['calibration'] = calibration_record
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Frozen 95th-percentile threshold:', calibration.threshold, 'clean validation rows:', calibration.validation_count)


## 7. Create three replay buffers once, then pair filter/no-filter arms

Conditions are clean replay, clean+gate, random 20%, random 20%+gate, targeted 20% old attacks→Benign, targeted 20%+gate. Each paired arm uses the **identical** supplied replay labels. Random flips choose a different ID from old IDs 0–4 and change 100/500. Targeted flips source IDs 1–4 to Benign ID 0 and change 80/400 eligible rows, only 16% of the full buffer. Both use seed 42 and half-up rounding. Original clean labels and changed positions are retained only for later audit, never passed to the gate.


In [ ]:
attacks = {
    'clean': apply_label_flip(replay.labels, 0.0, 42, mode='random', allowed_class_ids=TASK1_CLASS_IDS, original_row_indices=replay.row_indices),
    'random20': apply_label_flip(replay.labels, 0.20, 42, mode='random', allowed_class_ids=TASK1_CLASS_IDS, original_row_indices=replay.row_indices),
    'targeted20': apply_label_flip(replay.labels, 0.20, 42, mode='targeted', source_class_ids=[1, 2, 3, 4], target_class_id=0, allowed_class_ids=TASK1_CLASS_IDS, original_row_indices=replay.row_indices),
}
assert (attacks['random20'].changed_count, attacks['random20'].eligible_count) == (100, 500)
assert (attacks['targeted20'].changed_count, attacks['targeted20'].eligible_count) == (80, 400)
assert attacks['targeted20'].achieved_total_fraction == 0.16
for name, attack in attacks.items():
    assert np.array_equal(attack.changed_original_row_indices, replay.row_indices[attack.changed_indices])
    print(name, 'budget:', attack.changed_count, '/', attack.eligible_count, 'eligible; total replay fraction:', attack.achieved_total_fraction)
assert np.array_equal(replay.labels, attacks['clean'].original_labels)
condition_plan = [('clean', False), ('clean', True), ('random20', False), ('random20', True), ('targeted20', False), ('targeted20', True)]


## 8. Train six Task 2 conditions from matching expanded weights

For each condition, independently reload the same clean Task 1 checkpoint, expand IDs 5–7 with the same seed, verify the initial-state hash, reset training RNG, and fit on clean Task 2 rows plus its replay buffer. The filter function receives **only** replay features, supplied labels, frozen teacher probabilities/class IDs, and the calibrated threshold. It cannot see clean labels or changed-index masks. Those are used afterward by a separate audit function. The gate never restores labels. If no replay row survives, `None` is passed explicitly as replay and the condition becomes Task-2-only. All arms use the same clean seen-class validation pool for checkpoint selection; the test set is held out. Filtering changes replay row counts and optimizer steps, which are recorded.


In [ ]:
def state_hash(model):
    digest = hashlib.sha256()
    for key, value in sorted(model.network.state_dict().items()):
        digest.update(key.encode()); digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()
def attack_record(attack):
    return {'mode': attack.mode, 'requested_rate': attack.requested_rate, 'seed': attack.seed, 'budget_denominator': attack.budget_denominator, 'eligible_count': attack.eligible_count, 'changed_count': attack.changed_count, 'achieved_eligible_fraction': attack.achieved_eligible_fraction, 'achieved_total_fraction': attack.achieved_total_fraction, 'allowed_class_ids': list(attack.allowed_class_ids), 'source_class_ids': list(attack.source_class_ids) if attack.source_class_ids is not None else None, 'target_class_id': attack.target_class_id, 'rounding': 'floor(rate * eligible_count + 0.5)'}
condition_models = {}
condition_histories = {}
initial_hash = None
for attack_name, filtered in condition_plan:
    name = attack_name + ('_filtered' if filtered else '_unfiltered')
    folder = OUTPUT_DIR / name; folder.mkdir()
    attack = attacks[attack_name]
    supplied_labels = attack.poisoned_labels
    if filtered:
        gate = apply_label_consistency_gate(replay.features, supplied_labels, replay_teacher_probs, teacher.class_ids, calibration.threshold)
        retained_mask = gate.retained_mask
        replay_argument = gate.as_fit_replay()
        scores = gate.scores
    else:
        retained_mask = np.ones(len(supplied_labels), dtype=bool)
        replay_argument = (replay.features.copy(), supplied_labels.copy())
        scores = label_inconsistency_scores(replay_teacher_probs, supplied_labels, teacher.class_ids)
    audit = replay_gate_metrics(retained_mask, attack.changed_indices, replay.labels, supplied_labels, class_ids=TASK1_CLASS_IDS)
    retained_replay_rows = int(retained_mask.sum())
    assert (replay_argument is None) == (retained_replay_rows == 0)
    if replay_argument is not None:
        assert len(replay_argument[1]) == retained_replay_rows
    np.savez_compressed(folder / 'replay_audit.npz', original_rows=replay.row_indices, original_clean_labels=attack.original_labels, supplied_labels=supplied_labels, changed_positions=attack.changed_indices, changed_original_rows=attack.changed_original_row_indices, teacher_scores=scores, retained_mask=retained_mask)
    (folder / 'attack.json').write_text(json.dumps(attack_record(attack), indent=2), encoding='utf-8')
    (folder / 'gate_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
    model = TabularTransformerClassifier.load(task1_checkpoint, device='cuda')
    assert model.class_ids == TASK1_CLASS_IDS
    model.add_classes(TASK2_CLASS_IDS)
    assert model.class_ids == TASK1_CLASS_IDS + TASK2_CLASS_IDS
    current_hash = state_hash(model)
    if initial_hash is None:
        initial_hash = current_hash
    assert current_hash == initial_hash, f'{name}: expanded initial weights differ'
    torch.manual_seed(model_config.seed); torch.cuda.manual_seed_all(model_config.seed)
    total_training_rows = len(tasks.task2.train.labels) + retained_replay_rows
    torch.cuda.synchronize(); start = perf_counter()
    history = model.fit(tasks.task2.train.features, tasks.task2.train.labels, tasks.seen_validation.features, tasks.seen_validation.labels, replay=replay_argument, verbose=True)
    torch.cuda.synchronize(); train_seconds = perf_counter() - start
    assert model.class_ids == TASK1_CLASS_IDS + TASK2_CLASS_IDS
    assert history.best_epoch == int(np.argmin(history.validation_loss)) + 1
    assert np.isfinite(history.train_loss + history.validation_loss + history.validation_accuracy).all()
    checkpoint = folder / 'best_validation.pt'; model.save(checkpoint)
    steps_per_epoch = ceil(total_training_rows / model.config.batch_size)
    condition_models[name] = model; condition_histories[name] = history
    condition_manifest = {'attack': attack_record(attack), 'filtered': filtered, 'gate_audit': audit, 'retained_replay_rows': retained_replay_rows, 'empty_replay': replay_argument is None, 'task2_new_rows': len(tasks.task2.train.labels), 'training_rows_per_epoch': total_training_rows, 'epochs_completed': history.epochs_completed, 'optimizer_steps_per_epoch': steps_per_epoch, 'optimizer_steps_total': steps_per_epoch * history.epochs_completed, 'train_seconds': train_seconds, 'expanded_initial_sha256': current_hash, 'config': asdict(model.config), 'history': history.to_dict(), 'checkpoint': str(checkpoint.relative_to(OUTPUT_DIR))}
    manifest['conditions'][name] = condition_manifest
    (folder / 'config.json').write_text(json.dumps({'model': asdict(model.config), 'attack': attack_record(attack), 'gate_enabled': filtered, 'calibration': calibration_record, 'source_sha': checked_out_sha}, indent=2), encoding='utf-8')
    (folder / 'history.json').write_text(json.dumps(history.to_dict(), indent=2), encoding='utf-8')
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(name, 'replay retained:', retained_replay_rows, 'poison rejected:', audit['poison_rejected'], 'clean false rejected:', audit['clean_false_rejected'], 'epochs:', history.epochs_completed, 'steps:', condition_manifest['optimizer_steps_total'])


## 9. Evaluate only after checkpoint selection

Compute old/new/combined classification, per-class recall/F1, Benign FPR, old attack→Benign errors, and signed forgetting. The same old-class test rows and five-class macro-F1 definition are used before/after. Predictions into new classes still count as old-class errors. Check each saved checkpoint by reloading and requiring identical combined-test predictions. These are observed metrics only after Kaggle runs; do not tune the 95th percentile against them.


In [ ]:
ALL_IDS = TASK1_CLASS_IDS + TASK2_CLASS_IDS
def checked_predictions(model, part):
    probabilities = model.predict_proba(part.features)
    predictions = model.predict(part.features)
    assert probabilities.shape == (len(part.labels), len(model.class_ids))
    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5)
    assert np.array_equal(predictions, np.asarray(model.class_ids)[probabilities.argmax(axis=1)])
    return predictions
def focused_metrics(truth, predictions, focus_ids, *, benign=False, source_to_benign=False):
    kwargs = {'benign_class_id': 0 if benign else None}
    if source_to_benign:
        kwargs.update(source_class_ids=[1, 2, 3, 4], target_class_id=0)
    result = classification_metrics(truth, predictions, class_ids=ALL_IDS, class_names=tasks.class_names, **kwargs)
    result['focused_class_ids'] = list(focus_ids)
    result['focused_macro_f1'] = float(np.mean([result['per_class'][str(class_id)]['f1'] for class_id in focus_ids]))
    return result
for name, model in condition_models.items():
    folder = OUTPUT_DIR / name
    old_pred = checked_predictions(model, tasks.task1.test)
    new_pred = checked_predictions(model, tasks.task2.test)
    combined_pred = checked_predictions(model, tasks.combined_test)
    old_metrics = focused_metrics(tasks.task1.test.labels, old_pred, TASK1_CLASS_IDS, benign=True, source_to_benign=True)
    new_metrics = focused_metrics(tasks.task2.test.labels, new_pred, TASK2_CLASS_IDS)
    combined_metrics = classification_metrics(tasks.combined_test.labels, combined_pred, class_ids=ALL_IDS, class_names=tasks.class_names, benign_class_id=0, source_class_ids=[1, 2, 3, 4], target_class_id=0)
    forgetting = forgetting_metrics(tasks.task1.test.labels, task1_predictions, old_pred, old_class_ids=TASK1_CLASS_IDS, all_class_ids=ALL_IDS)
    reloaded = TabularTransformerClassifier.load(folder / 'best_validation.pt', device='cuda')
    reload_identical = reloaded.class_ids == model.class_ids and np.array_equal(reloaded.predict(tasks.combined_test.features), combined_pred)
    assert reload_identical, f'{name}: checkpoint reload predictions differ'
    metrics = {'task1_before': task1_before, 'task1_after': old_metrics, 'task2_after': new_metrics, 'combined_after': combined_metrics, 'forgetting': forgetting}
    (folder / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    manifest['conditions'][name]['metrics'] = metrics
    manifest['conditions'][name]['checkpoint_reload_identical'] = bool(reload_identical)
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(name, 'old:', old_metrics['accuracy'], 'new:', new_metrics['accuracy'], 'combined:', combined_metrics['accuracy'], 'old macro-F1:', old_metrics['focused_macro_f1'], 'benign FPR:', old_metrics['benign_false_positive']['rate'], 'attack-to-Benign:', old_metrics['source_to_target_misclassification']['rate'], 'forgetting:', forgetting['accuracy_forgetting'])


## 10. Compare paired attacks with both controls; save plots

Compare each filtered attack with its **identical** attacked-unfiltered buffer and with the clean-filtered control. Also retain clean-unfiltered as a reference. The gate can reject clean rows or retain poison; a score improvement is not assumed. Record training-work differences and inspect the plot before drawing conclusions. Send the whole timestamped output and executed notebook back for review.


In [ ]:
def outcome(name):
    record = manifest['conditions'][name]
    metrics = record['metrics']
    return {'old_accuracy': metrics['task1_after']['accuracy'], 'old_macro_f1': metrics['task1_after']['focused_macro_f1'], 'new_accuracy': metrics['task2_after']['accuracy'], 'combined_accuracy': metrics['combined_after']['accuracy'], 'combined_macro_f1': metrics['combined_after']['macro_f1'], 'benign_fpr': metrics['task1_after']['benign_false_positive']['rate'], 'attack_to_benign_rate': metrics['task1_after']['source_to_target_misclassification']['rate'], 'accuracy_forgetting': metrics['forgetting']['accuracy_forgetting'], 'retained_replay': record['retained_replay_rows'], 'optimizer_steps': record['optimizer_steps_total'], 'poison_rejection_rate': record['gate_audit']['poison_rejection_rate'], 'clean_false_rejection_rate': record['gate_audit']['clean_false_rejection_rate'], 'retained_poison_fraction': record['gate_audit']['retained_poison_fraction']}
outcomes = {name: outcome(name) for name in manifest['conditions']}
comparison = {}
for attack_name in ('clean', 'random20', 'targeted20'):
    unfiltered_name = attack_name + '_unfiltered'; filtered_name = attack_name + '_filtered'
    reference = outcomes[unfiltered_name]; filtered = outcomes[filtered_name]; clean_filtered = outcomes['clean_filtered']
    comparison[attack_name] = {'unfiltered': reference, 'filtered': filtered, 'filtered_minus_paired_unfiltered': {key: filtered[key] - reference[key] for key in ('old_accuracy', 'old_macro_f1', 'new_accuracy', 'combined_accuracy', 'combined_macro_f1', 'benign_fpr', 'attack_to_benign_rate', 'accuracy_forgetting')}, 'filtered_minus_clean_filtered': {key: filtered[key] - clean_filtered[key] for key in ('old_accuracy', 'old_macro_f1', 'new_accuracy', 'combined_accuracy', 'combined_macro_f1', 'benign_fpr', 'attack_to_benign_rate', 'accuracy_forgetting')}}
(OUTPUT_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
display(pd.DataFrame(outcomes).T)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for axis, (name, history) in zip(axes.flat, condition_histories.items()):
    epochs = np.arange(1, history.epochs_completed + 1)
    axis.plot(epochs, history.train_loss, label='Train loss')
    axis.plot(epochs, history.validation_loss, label='Clean validation loss')
    axis.axvline(history.best_epoch, color='gray', linestyle='--', label='Best epoch')
    axis.set(title=name, xlabel='Epoch', ylabel='Cross-entropy loss')
    axis.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160); plt.show()
manifest['expanded_initial_sha256_all_conditions'] = initial_hash
manifest['comparison_file'] = 'comparison.json'
manifest['plot_file'] = 'training_curves.png'
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Source SHA:', checked_out_sha, 'input SHA-256:', input_sha256, 'GPU:', environment['gpu_name'])
print('Six condition reloads passed; output folder:', OUTPUT_DIR)
